# BioSkills Lab — AI/ML Capstone
## Did the Foundation Model Actually Help?

**Course:** AI for Genomics in the Foundation Model Era  
**Dataset:** PBMC 3k (built-in scanpy) + scIB benchmark  
**Task:** Cell type annotation — comparing PCA, scVI, and Geneformer  

[![BioSkills Lab](https://img.shields.io/badge/BioSkills-Lab-38bdf8)](https://bioskillslab.dev)

---

### Runtime recommendation
- PCA + scVI: CPU or GPU both fine
- Geneformer: requires **GPU runtime** (Runtime > Change runtime type > T4 GPU)

> **Note:** Geneformer is optional. Steps 1-4 (PCA vs scVI) run fine on CPU.

In [ ]:
!pip install -q scanpy scvi-tools anndata scikit-learn umap-learn

In [ ]:
import scanpy as sc
import scvi
import numpy as np
import pandas as pd
import torch
import time
import warnings
warnings.filterwarnings('ignore')
import matplotlib.pyplot as plt
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score
from sklearn.preprocessing import LabelEncoder

print(f'scanpy: {sc.__version__}')
print(f'scvi-tools: {scvi.__version__}')
print(f'GPU available: {torch.cuda.is_available()}')

## Step 1: Load Dataset

We use the PBMC 3k dataset (built into scanpy, ~6MB). For a larger multi-donor benchmark, we attempt to load the scIB dataset from figshare.

In [ ]:
import urllib.request

# Try scIB benchmark dataset first (~150MB)
USE_SCIB = False
try:
    print('Attempting to download scIB PBMC benchmark...')
    urllib.request.urlretrieve('https://figshare.com/ndownloader/files/24539842', 'pbmc_scib.h5ad')
    adata = sc.read_h5ad('pbmc_scib.h5ad')
    if 'cell_type' not in adata.obs.columns:
        raise ValueError('cell_type column missing')
    USE_SCIB = True
    print(f'scIB dataset loaded: {adata}')
except Exception as e:
    print(f'scIB download failed ({e}) — using PBMC 3k built-in dataset')
    adata = sc.datasets.pbmc3k_processed()
    # pbmc3k_processed has louvain clusters, use those as proxy labels
    if 'louvain' in adata.obs.columns:
        adata.obs['cell_type'] = adata.obs['louvain']
    print(f'PBMC 3k loaded: {adata}')

print('\nCell types:')
print(adata.obs['cell_type'].value_counts())

## Step 2: Preprocessing

In [ ]:
# Store raw counts
if 'counts' not in adata.layers:
    adata.layers['counts'] = adata.X.copy()

# Highly variable genes
batch_key = 'batch' if 'batch' in adata.obs.columns else None
sc.pp.highly_variable_genes(adata, n_top_genes=2000, subset=False,
                              flavor='seurat_v3', batch_key=batch_key)

# Normalise for PCA
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)
sc.pp.scale(adata, max_value=10)

# Encode labels
le = LabelEncoder()
y = le.fit_transform(adata.obs['cell_type'])
indices = np.arange(len(y))
train_idx, test_idx = train_test_split(indices, test_size=0.2, random_state=42, stratify=y)
y_train, y_test = y[train_idx], y[test_idx]
print(f'Train: {len(train_idx)}, Test: {len(test_idx)}')
print(f'Cell types: {le.classes_}')

## Step 3: Method 1 — PCA + kNN

In [ ]:
t0 = time.time()
sc.tl.pca(adata, n_comps=50, use_highly_variable=True)
X_pca = adata.obsm['X_pca']
pca_time = time.time() - t0

knn = KNeighborsClassifier(n_neighbors=15, metric='cosine', n_jobs=-1)
knn.fit(X_pca[train_idx], y_train)
y_pred_pca = knn.predict(X_pca[test_idx])

acc_pca = accuracy_score(y_test, y_pred_pca)
f1_pca  = f1_score(y_test, y_pred_pca, average='macro')
print(f'PCA + kNN  |  Accuracy: {acc_pca:.3f}  |  Macro-F1: {f1_pca:.3f}  |  Time: {pca_time:.1f}s')

## Step 4: Method 2 — scVI + kNN

In [ ]:
t0 = time.time()
adata_scvi = adata.copy()
adata_scvi.X = adata_scvi.layers['counts']

setup_kwargs = {'layer': 'counts'}
if batch_key:
    setup_kwargs['batch_key'] = batch_key
scvi.model.SCVI.setup_anndata(adata_scvi, **setup_kwargs)
model_scvi = scvi.model.SCVI(adata_scvi, n_latent=20, n_layers=2)
model_scvi.train(max_epochs=200, early_stopping=True, early_stopping_patience=20, batch_size=256)

X_scvi = model_scvi.get_latent_representation()
adata.obsm['X_scVI'] = X_scvi
scvi_time = time.time() - t0

knn_scvi = KNeighborsClassifier(n_neighbors=15, metric='cosine', n_jobs=-1)
knn_scvi.fit(X_scvi[train_idx], y_train)
y_pred_scvi = knn_scvi.predict(X_scvi[test_idx])

acc_scvi = accuracy_score(y_test, y_pred_scvi)
f1_scvi  = f1_score(y_test, y_pred_scvi, average='macro')
print(f'scVI + kNN  |  Accuracy: {acc_scvi:.3f}  |  Macro-F1: {f1_scvi:.3f}  |  Time: {scvi_time:.1f}s')

## Step 5: Method 3 — Geneformer (GPU required)

> Switch to GPU runtime before running: **Runtime > Change runtime type > T4 GPU**  
> If you are on CPU, this cell will be skipped automatically.

In [ ]:
GENEFORMER_AVAILABLE = False
acc_gf, f1_gf, gf_time = 0.0, 0.0, 0.0

if not torch.cuda.is_available():
    print('No GPU detected — skipping Geneformer. Switch to T4 GPU runtime to run this step.')
else:
    try:
        print('Installing Geneformer...')
        import subprocess
        subprocess.run(['pip', 'install', '-q', 'geneformer'], check=True, capture_output=True)
        from geneformer import TranscriptomeTokenizer, EmbExtractor
        import os

        t0 = time.time()
        os.makedirs('gf_input', exist_ok=True)
        os.makedirs('gf_tokens', exist_ok=True)
        os.makedirs('gf_embs', exist_ok=True)

        adata_gf = adata.copy()
        adata_gf.X = adata_gf.layers['counts']
        import scipy.sparse as sp
        if sp.issparse(adata_gf.X):
            adata_gf.X = adata_gf.X.toarray()
        adata_gf.X = adata_gf.X.astype(int)
        adata_gf.write_loom('gf_input/pbmc.loom')

        tk = TranscriptomeTokenizer({'cell_type': 'cell_type'}, nproc=2)
        tk.tokenize_data('gf_input/', 'gf_tokens/', 'pbmc', file_format='loom')

        embex = EmbExtractor(model_type='Pretrained', num_classes=0,
                              emb_mode='cell', cell_emb_style='mean_pool',
                              emb_layer=-1, emb_label=['cell_type'],
                              forward_batch_size=100, nproc=2)
        embs = embex.extract_embs('ctheodoris/Geneformer', 'gf_tokens/pbmc.dataset', 'gf_embs/', 'pbmc')

        X_gf = embs.values
        adata.obsm['X_geneformer'] = X_gf
        gf_time = time.time() - t0

        knn_gf = KNeighborsClassifier(n_neighbors=15, metric='cosine', n_jobs=-1)
        knn_gf.fit(X_gf[train_idx], y_train)
        y_pred_gf = knn_gf.predict(X_gf[test_idx])

        acc_gf = accuracy_score(y_test, y_pred_gf)
        f1_gf  = f1_score(y_test, y_pred_gf, average='macro')
        GENEFORMER_AVAILABLE = True
        print(f'Geneformer + kNN  |  Accuracy: {acc_gf:.3f}  |  Macro-F1: {f1_gf:.3f}  |  Time: {gf_time:.1f}s')
    except Exception as e:
        print(f'Geneformer failed: {e}')
        print('Continuing with PCA and scVI results only.')

## Step 6: Final Comparison

In [ ]:
rows = [
    {'Method': 'PCA + kNN',  'Accuracy': acc_pca,  'Macro-F1': f1_pca,  'Time (s)': pca_time},
    {'Method': 'scVI + kNN', 'Accuracy': acc_scvi, 'Macro-F1': f1_scvi, 'Time (s)': scvi_time},
]
if GENEFORMER_AVAILABLE:
    rows.append({'Method': 'Geneformer + kNN', 'Accuracy': acc_gf, 'Macro-F1': f1_gf, 'Time (s)': gf_time})

results = pd.DataFrame(rows).sort_values('Macro-F1', ascending=False)
print('\n' + '='*55)
print('FINAL RESULTS: Did the Foundation Model Actually Help?')
print('='*55)
print(results.to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
colors = ['#38bdf8', '#4ade80', '#a78bfa'][:len(results)]
axes[0].bar(results['Method'], results['Macro-F1'], color=colors)
axes[0].set_ylabel('Macro-F1'); axes[0].set_title('Classification Performance')
axes[0].tick_params(axis='x', rotation=15)
axes[1].bar(results['Method'], results['Time (s)'], color=colors)
axes[1].set_ylabel('Runtime (seconds)'); axes[1].set_title('Computational Cost')
axes[1].tick_params(axis='x', rotation=15)
plt.tight_layout(); plt.show()

## Your Answer

```
RESULTS:
  PCA Macro-F1:         ___
  scVI Macro-F1:        ___
  Geneformer Macro-F1:  ___ (if run)

CONCLUSION:
  Did scVI outperform PCA? By how much?
  Was the gain worth the extra runtime?
  Which method would you use in a real analysis?
  What surprised you most?
```

---
**BioSkills Lab** — bioskillslab.dev